In [54]:
from langgraph.graph import StateGraph
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from typing import TypedDict
from langgraph.constants import END, START

In [55]:
load_dotenv()


True

In [56]:
class Batsman(TypedDict):

    runs: int
    fours: int
    sixes: int
    balls: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [57]:
def calculate_sr(state: Batsman) -> Batsman:

    sr = (state['runs']/state['balls'])*100

    return {'sr' : sr}

In [58]:
def calculate_bpb(state: Batsman) -> Batsman:

    bpb = (state['balls']) / (state['fours'] + state['sixes'])
    
    return {'bpb': bpb}

In [59]:
def calculate_boundary_percent(state: Batsman) -> Batsman:

    boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6)) / (state['runs']))*100
    
    return {'boundary_percent': boundary_percent}

In [60]:
def summary(state: Batsman) -> Batsman:

    summary = f"""
    Strike rate = {state['sr']} \n
    BPB = {state['bpb']} \n
    Boundary percentage = {state['boundary_percent']}
    """

    return {'summary': summary}

In [61]:
graph = StateGraph(Batsman)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()

In [62]:
initial_state = {
    'runs': 90,
    'balls': 40,
    'fours': 6,
    'sixes': 5
}

workflow.invoke(initial_state)

{'runs': 90,
 'fours': 6,
 'sixes': 5,
 'balls': 40,
 'sr': 225.0,
 'bpb': 3.6363636363636362,
 'boundary_percent': 60.0,
 'summary': '\n    Strike rate = 225.0 \n\n    BPB = 3.6363636363636362 \n\n    Boundary percentage = 60.0\n    '}